# Algoritmos básicos · Cálculo numérico

Métodos numéricos clásicos programados a mano: derivadas por diferencias finitas, raíces con Herón, Newton y bisección, funciones inversas a partir de un buscador de ceros, integración por rectángulos, trapecios y Simpson, límites que se aproximan con sucesiones y series, sistemas lineales por eliminación gaussiana y determinantes. Se presta atención al error: cómo baja al refinar y cuándo los redondeos de los `float` lo hacen subir. Al final, conversiones exactas con `Fraction` entre decimales periódicos y fracciones. Cada resultado se compara con `math`, `numpy` o un cálculo exacto. Ideas de los ejercicios de [José A. Alonso](https://github.com/jaalonso/Ejercicios-Python) y de [Edabit](https://edabit.com/es/challenges/python3).

In [1]:
import math
import random
import re
from decimal import Decimal, getcontext
from fractions import Fraction

import numpy as np

## 1. Derivada numérica y el paso $h$

Aproximar $f'(x)$ con las diferencias hacia delante $\frac{f(x+h) - f(x)}{h}$, hacia atrás $\frac{f(x) - f(x-h)}{h}$ y centrada $\frac{f(x+h) - f(x-h)}{2h}$. Para $f = \sin$ en $x = 1$, mostrar una tabla con el error de cada una frente a $\cos 1$ para $h = 10^{-1}, \dots, 10^{-13}$. Al bajar $h$ el error de truncamiento disminuye ($O(h)$ en las laterales, $O(h^2)$ en la centrada), pero el numerador resta dos números casi iguales y su error de redondeo, del orden de $\varepsilon / h$ con $\varepsilon \approx 2{,}2 \cdot 10^{-16}$, crece: comprobar que el mejor $h$ ronda $\sqrt{\varepsilon} \approx 10^{-8}$ en las laterales y $\sqrt[3]{\varepsilon} \approx 10^{-5}$ en la centrada.

In [2]:
def adelante(f, x, h):
    return (f(x + h) - f(x)) / h


def atras(f, x, h):
    return (f(x) - f(x - h)) / h


def centrada(f, x, h):
    return (f(x + h) - f(x - h)) / (2 * h)


exacta = math.cos(1)
metodos = {"adelante": adelante, "atrás": atras, "centrada": centrada}
errores = {nombre: {} for nombre in metodos}
print(f"{'h':>7}" + "".join(f"{nombre:>11}" for nombre in metodos))
for k in range(1, 14):
    h = 10.0**-k
    for nombre, metodo in metodos.items():
        errores[nombre][k] = abs(metodo(math.sin, 1, h) - exacta)
    print(f"{h:>7.0e}" + "".join(f"{errores[nombre][k]:>11.1e}" for nombre in metodos))

mejor = {nombre: min(e, key=e.get) for nombre, e in errores.items()}
print("Mejor exponente k (h = 10⁻ᵏ):", mejor)
assert 7 <= mejor["adelante"] <= 9 and 7 <= mejor["atrás"] <= 9 and 4 <= mejor["centrada"] <= 6
assert all(e[13] > 100 * e[mejor[nombre]] for nombre, e in errores.items())

      h   adelante      atrás   centrada
  1e-01    4.3e-02    4.1e-02    9.0e-04
  1e-02    4.2e-03    4.2e-03    9.0e-06
  1e-03    4.2e-04    4.2e-04    9.0e-08
  1e-04    4.2e-05    4.2e-05    9.0e-10
  1e-05    4.2e-06    4.2e-06    1.1e-11
  1e-06    4.2e-07    4.2e-07    2.8e-11
  1e-07    4.2e-08    4.1e-08    1.9e-10
  1e-08    3.0e-09    8.1e-09    2.6e-09
  1e-09    5.3e-08    5.8e-08    3.0e-09
  1e-10    5.8e-08    5.8e-08    5.8e-08
  1e-11    1.2e-06    1.2e-06    1.2e-06
  1e-12    4.3e-05    6.8e-05    1.2e-05
  1e-13    7.3e-04    3.8e-04    1.8e-04
Mejor exponente k (h = 10⁻ᵏ): {'adelante': 8, 'atrás': 8, 'centrada': 5}


## 2. Raíz cuadrada por el método de Herón

Para calcular $\sqrt{a}$, Herón de Alejandría partía de una aproximación $x$ y la sustituía por la media de $x$ y $a / x$ (uno se pasa y el otro se queda corto). Mostrar las iteraciones para $\sqrt 2$ con `Decimal` a 60 cifras, contando las cifras correctas en cada paso (se duplican, porque es el método de Newton para $x^2 - a$), y comprobar la versión con `float` contra `math.sqrt` para muchos valores.

In [3]:
def heron(a, x=None, tolerancia=1e-15):
    if a == 0:
        return a
    x = x or (a if a > 1 else 1)
    while True:
        siguiente = (x + a / x) / 2
        if abs(siguiente - x) <= tolerancia * siguiente:
            return siguiente
        x = siguiente


getcontext().prec = 60
a, x = Decimal(2), Decimal(1)
real = a.sqrt()
print(f"{'paso':>4} {'cifras correctas':>17}  aproximación")
for paso in range(1, 8):
    x = (x + a / x) / 2
    correctas = -int(abs(x - real).log10()) if x != real else 60
    print(f"{paso:>4} {correctas:>17}  {str(x)[:42]}")

random.seed(2)
valores = [0, 1e-10, 0.5, 2, 10, 1e6, 1e20] + [random.uniform(0, 1000) for _ in range(1000)]
assert all(math.isclose(heron(v), math.sqrt(v), rel_tol=1e-14) for v in valores)

paso  cifras correctas  aproximación
   1                 1  1.5
   2                 2  1.4166666666666666666666666666666666666666
   3                 5  1.4142156862745098039215686274509803921568
   4                11  1.4142135623746899106262955788901349101165
   5                24  1.4142135623730950488016896235025302436149
   6                48  1.4142135623730950488016887242096980785696
   7                60  1.4142135623730950488016887242096980785696


## 3. Método de Newton genérico

Escribir `newton(f, x0, df=None)`, que aplique $x \leftarrow x - f(x) / f'(x)$ hasta que el paso sea despreciable, usando la derivada exacta si se da y la derivada numérica centrada si no, y que devuelva la raíz y las iteraciones. Resolver $\cos x = x$, $x^3 - 2x - 5 = 0$ y $e^x = 3$ con las dos derivadas, comprobando las raíces. Ver también un caso en el que Newton no converge: $f(x) = \sqrt[3]{x}$, donde cada paso dobla la distancia a la raíz.

In [4]:
def derivada(f, x, h=1e-6):
    return (f(x + h) - f(x - h)) / (2 * h)


def newton(f, x0, df=None, tolerancia=1e-13, max_iteraciones=60):
    x = x0
    for i in range(1, max_iteraciones + 1):
        d = df(x) if df else derivada(f, x)
        paso = f(x) / d
        x -= paso
        if abs(paso) <= tolerancia * (1 + abs(x)):
            return x, i
    raise ArithmeticError(f"sin convergencia en {max_iteraciones} iteraciones (x = {x:.3g})")


problemas = {
    "cos x = x": (lambda x: math.cos(x) - x, lambda x: -math.sin(x) - 1, 1.0),
    "x³ - 2x - 5 = 0": (lambda x: x**3 - 2 * x - 5, lambda x: 3 * x**2 - 2, 2.0),
    "eˣ = 3": (lambda x: math.exp(x) - 3, math.exp, 0.0),
}
print(f"{'ecuación':>16} {'raíz':>20} {'iter. exacta':>13} {'iter. numérica':>15}")
for nombre, (f, df, x0) in problemas.items():
    raiz, i_exacta = newton(f, x0, df)
    raiz_num, i_num = newton(f, x0)
    print(f"{nombre:>16} {raiz:>20.15f} {i_exacta:>13} {i_num:>15}")
    assert abs(f(raiz)) < 1e-12 and math.isclose(raiz, raiz_num, rel_tol=1e-9)
assert math.isclose(newton(lambda x: math.exp(x) - 3, 0.0)[0], math.log(3))

try:
    newton(math.cbrt, 0.1, max_iteraciones=20)
except ArithmeticError as e:
    print("Raíz cúbica:", e)

        ecuación                 raíz  iter. exacta  iter. numérica
       cos x = x    0.739085133215161             5               5
 x³ - 2x - 5 = 0    2.094551481542327             5               5
          eˣ = 3    1.098612288668110             7               7
Raíz cúbica: sin convergencia en 20 iteraciones (x = 1.05e+05)


## 4. Funciones inversas con un buscador de ceros

Escribir `cero(f, a, b)`, un buscador de ceros que combine la seguridad de la bisección con la rapidez de Newton: mantiene un intervalo $[a, b]$ donde $f$ cambia de signo, prueba un paso de Newton con la derivada numérica y, si se sale del intervalo, bisecta. Con él, escribir `inversa(f, a, b)`, que devuelva la función inversa de una $f$ monótona en $[a, b]$, y construir la raíz cúbica, el arcoseno y el logaritmo a partir de $x^3$, $\sin$ y $\exp$. Compararlas con `math.cbrt`, `math.asin` y `math.log`.

In [5]:
def cero(f, a, b, tolerancia=1e-15):
    fa, fb = f(a), f(b)
    if fa == 0:
        return a
    if fb == 0:
        return b
    if (fa < 0) == (fb < 0):
        raise ValueError("f no cambia de signo en el intervalo")
    x = (a + b) / 2
    for _ in range(300):
        fx = f(x)
        if fx == 0:
            return x
        if (fx < 0) == (fa < 0):
            a, fa = x, fx
        else:
            b = x
        d = derivada(f, x)
        siguiente = x - fx / d if d else (a + b) / 2
        if not a < siguiente < b:
            siguiente = (a + b) / 2
        if abs(siguiente - x) <= tolerancia * (1 + abs(x)):
            return siguiente
        x = siguiente
    return x


def inversa(f, a, b):
    return lambda y: cero(lambda x: f(x) - y, a, b)


raiz_cubica = inversa(lambda x: x**3, -11, 11)
arcoseno = inversa(math.sin, -math.pi / 2, math.pi / 2)
logaritmo = inversa(math.exp, -50, 50)

casos = [
    ("raíz cúbica", raiz_cubica, math.cbrt, np.linspace(-1000, 1000, 401)),
    ("arcoseno", arcoseno, math.asin, np.linspace(-1, 1, 401)),
    ("logaritmo", logaritmo, math.log, np.geomspace(1e-20, 1e20, 401)),
]
print(f"{'función':>12} {'error máximo':>13}")
for nombre, propia, referencia, ys in casos:
    error = max(abs(propia(float(y)) - referencia(float(y))) for y in ys)
    print(f"{nombre:>12} {error:>13.1e}")
    assert error < 1e-7
print("arcoseno(0.5) * 6 =", arcoseno(0.5) * 6)
try:
    arcoseno(1.5)
except ValueError as e:
    print("arcoseno(1.5):", e)

     función  error máximo
 raíz cúbica       1.2e-14
    arcoseno       2.2e-16
   logaritmo       5.0e-14
arcoseno(0.5) * 6 = 3.1415926535897936
arcoseno(1.5): f no cambia de signo en el intervalo


## 5. Raíz cuadrada por búsqueda binaria en los reales

Calcular $\sqrt a$ con una búsqueda binaria sobre el intervalo $[0, \max(1, a)]$: si $m^2 \le a$ la raíz está a la derecha de $m$. Con una precisión $\varepsilon$ hacen falta $\lceil \log_2 (\text{longitud} / \varepsilon) \rceil$ iteraciones; comprobarlo. Hacer después una versión sin $\varepsilon$ que siga hasta que el punto medio coincida con un extremo (los dos extremos son `float` consecutivos) y comprobar que `math.sqrt` cae entre ellos. Por último, la versión entera para números enormes, contra `math.isqrt`.

In [6]:
def raiz_biseccion(a, epsilon):
    bajo, alto = 0.0, max(1.0, a)
    iteraciones = 0
    while alto - bajo > epsilon:
        medio = (bajo + alto) / 2
        if medio * medio <= a:
            bajo = medio
        else:
            alto = medio
        iteraciones += 1
    return (bajo + alto) / 2, iteraciones


def raiz_biseccion_total(a):
    bajo, alto = 0.0, max(1.0, a)
    while (medio := (bajo + alto) / 2) not in (bajo, alto):
        if medio * medio <= a:
            bajo = medio
        else:
            alto = medio
    return bajo, alto


print(f"{'a':>6} {'ε':>7} {'raíz':>18} {'iter.':>6} {'previstas':>9}")
for a in (2, 10, 1000):
    for epsilon in (1e-3, 1e-9):
        raiz, iteraciones = raiz_biseccion(a, epsilon)
        previstas = math.ceil(math.log2(max(1, a) / epsilon))
        print(f"{a:>6} {epsilon:>7.0e} {raiz:>18.12f} {iteraciones:>6} {previstas:>9}")
        assert iteraciones == previstas and abs(raiz - math.sqrt(a)) <= epsilon

random.seed(4)
for a in [0.25, 2.0, 1e-8, 1e12] + [random.uniform(0, 1e6) for _ in range(500)]:
    bajo, alto = raiz_biseccion_total(a)
    assert bajo <= math.sqrt(a) <= alto and math.nextafter(bajo, math.inf) == alto


def isqrt_biseccion(n):
    bajo, alto = 0, 1 << ((n.bit_length() + 1) // 2 + 1)
    while alto - bajo > 1:
        medio = (bajo + alto) // 2
        if medio * medio <= n:
            bajo = medio
        else:
            alto = medio
    return bajo


grandes = [10**40, 2**127 - 1] + [random.getrandbits(300) for _ in range(200)]
assert all(isqrt_biseccion(n) == math.isqrt(n) for n in grandes + list(range(2000)))
print("⌊√(10⁴⁰ + 12345)⌋ =", isqrt_biseccion(10**40 + 12345))

     a       ε               raíz  iter. previstas
     2   1e-03     1.414550781250     11        11
     2   1e-09     1.414213561919     31        31
    10   1e-03     3.162536621094     14        14
    10   1e-09     3.162277660158     34        34
  1000   1e-03    31.622409820557     20        20
  1000   1e-09    31.622776602035     40        40
⌊√(10⁴⁰ + 12345)⌋ = 100000000000000000000


## 6. Integración numérica

Aproximar $\int_a^b f(x)\,dx$ dividiendo $[a, b]$ en $n$ subintervalos con las reglas de los rectángulos (por la izquierda, por la derecha y en el punto medio), los trapecios y Simpson ($n$ par). Para $\int_0^1 e^x\,dx = e - 1$, mostrar el error de cada regla con $n = 4, 8, \dots, 256$ y el cociente entre errores consecutivos: al duplicar $n$ el error se divide por 2 en los rectángulos laterales (orden 1), por 4 en el punto medio y los trapecios (orden 2) y por 16 en Simpson (orden 4).

In [7]:
def rect_izquierda(f, a, b, n):
    h = (b - a) / n
    return h * sum(f(a + i * h) for i in range(n))


def rect_derecha(f, a, b, n):
    h = (b - a) / n
    return h * sum(f(a + i * h) for i in range(1, n + 1))


def punto_medio(f, a, b, n):
    h = (b - a) / n
    return h * sum(f(a + (i + 0.5) * h) for i in range(n))


def trapecios(f, a, b, n):
    h = (b - a) / n
    return h * ((f(a) + f(b)) / 2 + sum(f(a + i * h) for i in range(1, n)))


def simpson(f, a, b, n):
    if n % 2:
        raise ValueError("Simpson necesita un número par de subintervalos")
    h = (b - a) / n
    impares = sum(f(a + i * h) for i in range(1, n, 2))
    pares = sum(f(a + i * h) for i in range(2, n, 2))
    return h / 3 * (f(a) + f(b) + 4 * impares + 2 * pares)


reglas = {"izquierda": rect_izquierda, "derecha": rect_derecha, "medio": punto_medio,
          "trapecios": trapecios, "Simpson": simpson}
exacto = math.e - 1
ns = [4 * 2**i for i in range(7)]
error = {nombre: [abs(regla(math.exp, 0, 1, n) - exacto) for n in ns] for nombre, regla in reglas.items()}
print(f"{'n':>4}" + "".join(f"{nombre:>11}" for nombre in reglas))
for i, n in enumerate(ns):
    print(f"{n:>4}" + "".join(f"{error[nombre][i]:>11.2e}" for nombre in reglas))
cocientes = {nombre: e[-2] / e[-1] for nombre, e in error.items()}
print("Cociente al pasar de 128 a 256:", {k: round(v, 2) for k, v in cocientes.items()})
for nombre, esperado in {"izquierda": 2, "derecha": 2, "medio": 4, "trapecios": 4, "Simpson": 16}.items():
    assert math.isclose(cocientes[nombre], esperado, rel_tol=0.05)

   n  izquierda    derecha      medio  trapecios    Simpson
   4   2.06e-01   2.24e-01   4.47e-03   8.94e-03   3.70e-05
   8   1.05e-01   1.10e-01   1.12e-03   2.24e-03   2.33e-06
  16   5.31e-02   5.43e-02   2.80e-04   5.59e-04   1.46e-07
  32   2.67e-02   2.70e-02   6.99e-05   1.40e-04   9.10e-09
  64   1.34e-02   1.35e-02   1.75e-05   3.50e-05   5.69e-10
 128   6.70e-03   6.72e-03   4.37e-06   8.74e-06   3.56e-11
 256   3.35e-03   3.36e-03   1.09e-06   2.18e-06   2.22e-12
Cociente al pasar de 128 a 256: {'izquierda': 2.0, 'derecha': 2.0, 'medio': 4.0, 'trapecios': 4.0, 'Simpson': 16.0}


## 7. Límites: ¿cuántos términos hacen falta?

Para cada error $10^{-k}$, encontrar el menor $n$ que lo consigue en tres límites:

- $\left(1 + \frac1n\right)^n \to e$. La sucesión crece, así que el menor $n$ se encuentra con búsqueda binaria. Calcularla como $e^{n \log(1 + 1/n)}$ con `math.log1p`: con `(1 + 1/n) ** n`, para $n$ muy grande $1 + 1/n$ ya se redondea y el resultado es absurdo.
- $\frac{\sin(1/n)}{1/n} \to 1$, también con búsqueda binaria (el error es $\approx \frac{1}{6n^2}$).
- La serie de Leibniz $4 \sum_{i=0}^{n-1} \frac{(-1)^i}{2i + 1} \to \pi$, sumando términos hasta alcanzar el error.

In [8]:
def menor_n(error, limite=10**12):
    # error(n) decrece con n
    bajo, alto = 1, limite
    while bajo < alto:
        medio = (bajo + alto) // 2
        if error(medio):
            alto = medio
        else:
            bajo = medio + 1
    return bajo


def aprox_e(n):
    return math.exp(n * math.log1p(1 / n))


for n in (10**6, 10**12, 10**15, 10**16):
    print(f"n = {n:.0e}: (1 + 1/n)**n = {(1 + 1 / n) ** n:.10f}   con log1p = {aprox_e(n):.10f}")

leibniz, suma, i = {}, 0.0, 0
for k in range(1, 7):
    while abs(4 * suma - math.pi) >= 10.0**-k:
        suma += (-1) ** i / (2 * i + 1)
        i += 1
    leibniz[k] = i

print(f"\n{'error':>6} {'n para e':>10} {'n para sen':>11} {'n para π':>10}")
for k in range(1, 7):
    cota = 10.0**-k
    n_e = menor_n(lambda n: math.e - aprox_e(n) < cota)
    n_sen = menor_n(lambda n: 1 - n * math.sin(1 / n) < cota)
    print(f"{cota:>6.0e} {n_e:>10} {n_sen:>11} {leibniz[k]:>10}")
    assert math.e - aprox_e(n_e) < cota <= math.e - aprox_e(n_e - 1)
    assert math.isclose(n_e, math.e / (2 * cota), rel_tol=0.1) or k < 3
    assert abs(n_sen - math.sqrt(1 / (6 * cota))) <= 1
    assert leibniz[k] <= 2 / cota

n = 1e+06: (1 + 1/n)**n = 2.7182804691   con log1p = 2.7182804693
n = 1e+12: (1 + 1/n)**n = 2.7185234960   con log1p = 2.7182818285
n = 1e+15: (1 + 1/n)**n = 3.0350352065   con log1p = 2.7182818285
n = 1e+16: (1 + 1/n)**n = 1.0000000000   con log1p = 2.7182818285



 error   n para e  n para sen   n para π
 1e-01         13           2         10
 1e-02        135           5        100
 1e-03       1359          13       1000
 1e-04      13591          41      10000
 1e-05     135914         130     100001
 1e-06    1359140         409    1000001


## 8. Sistema triangular: sustitución regresiva

Resolver $Ux = b$ con $U$ triangular superior despejando de la última ecuación a la primera: $x_i = \left(b_i - \sum_{j > i} u_{ij} x_j\right) / u_{ii}$. Probarlo con `Fraction` para obtener la solución exacta de un sistema pequeño y con una matriz aleatoria de $8 \times 8$ contra `numpy.linalg.solve`. Si algún $u_{ii}$ es 0, el sistema no tiene solución única: lanzar un `ValueError`.

In [9]:
def sustitucion_regresiva(U, b):
    n = len(b)
    x = [0] * n
    for i in reversed(range(n)):
        if U[i][i] == 0:
            raise ValueError(f"pivote nulo en la fila {i}")
        x[i] = (b[i] - sum(U[i][j] * x[j] for j in range(i + 1, n))) / U[i][i]
    return x


U = [[Fraction(v) for v in fila] for fila in [[2, -1, 3], [0, 4, 1], [0, 0, 6]]]
b = [Fraction(5), Fraction(2), Fraction(3)]
x = sustitucion_regresiva(U, b)
print("Solución exacta:", [str(v) for v in x])
assert all(sum(U[i][j] * x[j] for j in range(3)) == b[i] for i in range(3))

rng = np.random.default_rng(8)
U = np.triu(rng.uniform(-5, 5, (8, 8))) + 10 * np.eye(8)
b = rng.uniform(-5, 5, 8)
assert np.allclose(sustitucion_regresiva(U.tolist(), b.tolist()), np.linalg.solve(U, b))
try:
    sustitucion_regresiva([[1, 2], [0, 0]], [3, 4])
except ValueError as e:
    print("Error:", e)

Solución exacta: ['31/16', '3/8', '1/2']
Error: pivote nulo en la fila 1


## 9. Eliminación gaussiana con pivoteo parcial

Resolver $Ax = b$ convirtiendo la matriz ampliada en triangular superior (restando a cada fila un múltiplo de la fila del pivote) y aplicando la sustitución regresiva. Con **pivoteo parcial**, antes de eliminar la columna $k$ se sube la fila con el mayor $|a_{ik}|$, para no dividir por números pequeños. Resolver un problema con datos inventados, comprobar sistemas aleatorios contra `numpy.linalg.solve` y mostrar con $\begin{pmatrix} 10^{-20} & 1 \\ 1 & 1 \end{pmatrix} x = \begin{pmatrix} 1 \\ 2 \end{pmatrix}$ (solución $\approx (1, 1)$) qué pasa sin pivoteo.

In [10]:
def gauss(A, b, pivoteo=True):
    n = len(b)
    M = [list(fila) + [bi] for fila, bi in zip(A, b)]
    for k in range(n):
        if pivoteo:
            p = max(range(k, n), key=lambda i: abs(M[i][k]))
            M[k], M[p] = M[p], M[k]
        if M[k][k] == 0:
            raise ValueError("el sistema no tiene solución única")
        for i in range(k + 1, n):
            factor = M[i][k] / M[k][k]
            for j in range(k, n + 1):
                M[i][j] -= factor * M[k][j]
    return sustitucion_regresiva([fila[:n] for fila in M], [fila[n] for fila in M])


# Tres compras en una frutería: kilos de papas, plátanos y tomates y el total pagado
A = [[3, 2, 1], [1, 4, 2], [2, 1, 3]]
totales = [8.6, 11.2, 8.4]
papas, platanos, tomates = gauss(A, totales)
print(f"Precio por kilo: papas {papas:.2f} €, plátanos {platanos:.2f} €, tomates {tomates:.2f} €")
assert np.allclose([papas, platanos, tomates], np.linalg.solve(A, totales))
assert np.allclose([papas, platanos, tomates], [1.2, 1.8, 1.4])

rng = np.random.default_rng(9)
for n in range(2, 30):
    A, b = rng.uniform(-10, 10, (n, n)), rng.uniform(-10, 10, n)
    assert np.allclose(gauss(A.tolist(), b.tolist()), np.linalg.solve(A, b))

A, b = [[1e-20, 1], [1, 1]], [1, 2]
print("Con pivoteo:", gauss(A, b), "  sin pivoteo:", gauss(A, b, pivoteo=False))
try:
    gauss([[1, 2], [2, 4]], [3, 6])
except ValueError as e:
    print("Sistema singular:", e)

Precio por kilo: papas 1.20 €, plátanos 1.80 €, tomates 1.40 €
Con pivoteo: [1.0, 1.0]   sin pivoteo: [0.0, 1.0]
Sistema singular: el sistema no tiene solución única


## 10. Determinante por cofactores y por eliminación

Calcular el determinante de dos formas: desarrollando por la primera fila, $\det A = \sum_j (-1)^j a_{0j} \det A_{0j}$ (recursivo, $O(n!)$), y por eliminación gaussiana, como el producto de los pivotes cambiado de signo en cada intercambio de filas ($O(n^3)$). Con `Fraction`, la eliminación es exacta y debe coincidir con los cofactores en matrices enteras; con `float`, comparar con `numpy.linalg.det`. Contar las multiplicaciones de cada método según $n$.

In [11]:
def det_cofactores(A, cuenta=None):
    if len(A) == 1:
        return A[0][0]
    total = 0
    for j, a in enumerate(A[0]):
        menor = [fila[:j] + fila[j + 1 :] for fila in A[1:]]
        if cuenta is not None:
            cuenta[0] += 1
        total += (-1) ** j * a * det_cofactores(menor, cuenta)
    return total


def det_eliminacion(A, cuenta=None):
    M = [list(fila) for fila in A]
    n, det = len(M), 1
    for k in range(n):
        p = max(range(k, n), key=lambda i: abs(M[i][k]))
        if M[p][k] == 0:
            return 0
        if p != k:
            M[k], M[p] = M[p], M[k]
            det = -det
        det *= M[k][k]
        for i in range(k + 1, n):
            factor = M[i][k] / M[k][k]
            for j in range(k, n):
                M[i][j] -= factor * M[k][j]
            if cuenta is not None:
                cuenta[0] += n - k
    return det


random.seed(10)
for n in range(1, 7):
    for _ in range(20):
        A = [[random.randint(-9, 9) for _ in range(n)] for _ in range(n)]
        exacto = det_cofactores(A)
        assert det_eliminacion([[Fraction(v) for v in fila] for fila in A]) == exacto
        assert math.isclose(det_eliminacion(A), np.linalg.det(A), rel_tol=1e-9, abs_tol=1e-6)
        assert round(np.linalg.det(A)) == exacto

print(f"{'n':>2} {'cofactores':>11} {'eliminación':>12}")
for n in range(2, 10):
    A = np.random.default_rng(n).uniform(-1, 1, (n, n)).tolist()
    c1, c2 = [0], [0]
    d1, d2 = det_cofactores(A, c1), det_eliminacion(A, c2)
    assert math.isclose(d1, d2, rel_tol=1e-9)
    print(f"{n:>2} {c1[0]:>11} {c2[0]:>12}")

 n  cofactores  eliminación
 2           2            2
 3           9            8
 4          40           20
 5         205           40
 6        1236           70
 7        8659          112
 8       69280          168


 9      623529          240


## 11. De decimal periódico a fracción

Convertir un decimal periódico escrito con el periodo entre paréntesis (`0.(3)`, `1.2(45)`, `-3.1(6)`) en una fracción exacta. Si la parte entera es $e$, el anteperiodo $A$ tiene $a$ cifras y el periodo $P$ tiene $p$ cifras, el número vale $e + \frac{A}{10^a} + \frac{P}{10^a (10^p - 1)}$. Rechazar con un `ValueError` las cadenas mal formadas y comprobar casos conocidos, como `0.(9)` $= 1$.

In [12]:
DECIMAL_PERIODICO = re.compile(r"(-?)(\d+)(?:\.(\d*)(?:\((\d+)\))?)?")


def a_fraccion(texto):
    m = DECIMAL_PERIODICO.fullmatch(texto.strip())
    if not m:
        raise ValueError(f"decimal no válido: {texto!r}")
    signo, entero, ante, periodo = m.groups()
    ante = ante or ""
    valor = Fraction(int(entero)) + Fraction(int(ante or 0), 10 ** len(ante))
    if periodo:
        valor += Fraction(int(periodo), 10 ** len(ante) * (10 ** len(periodo) - 1))
    return -valor if signo else valor


ejemplos = {"0.(3)": Fraction(1, 3), "1.2(45)": Fraction(137, 110), "-3.1(6)": Fraction(-19, 6),
            "0.(9)": Fraction(1), "0.(142857)": Fraction(1, 7), "2.5": Fraction(5, 2), "7": Fraction(7),
            "0.08(3)": Fraction(1, 12)}
for texto, esperado in ejemplos.items():
    print(f"{texto:>11} = {a_fraccion(texto)}")
    assert a_fraccion(texto) == esperado
for malo in ("1.2(", "1..2", "(3)", "0.(3)4"):
    try:
        a_fraccion(malo)
    except ValueError as e:
        print(e)

      0.(3) = 1/3
    1.2(45) = 137/110
    -3.1(6) = -19/6
      0.(9) = 1
 0.(142857) = 1/7
        2.5 = 5/2
          7 = 7
    0.08(3) = 1/12
decimal no válido: '1.2('
decimal no válido: '1..2'
decimal no válido: '(3)'
decimal no válido: '0.(3)4'


## 12. De fracción a decimal periódico

Hacer la operación inversa con la división larga: cada nueva cifra sale del resto anterior multiplicado por 10, y en cuanto un resto se repite, las cifras desde su primera aparición forman el periodo. Comprobar el viaje de ida y vuelta con el ejercicio anterior para todas las fracciones $p/q$ con $q \le 60$, y que la longitud del periodo de $1/q$, con $q$ primo con 10, es el orden multiplicativo de 10 módulo $q$ (el menor $k$ con $10^k \equiv 1$).

In [13]:
def a_decimal(fraccion):
    fraccion = Fraction(fraccion)
    signo = "-" if fraccion < 0 else ""
    entero, resto = divmod(abs(fraccion.numerator), fraccion.denominator)
    cifras_dec, posicion = [], {}
    while resto and resto not in posicion:
        posicion[resto] = len(cifras_dec)
        cifra, resto = divmod(resto * 10, fraccion.denominator)
        cifras_dec.append(str(cifra))
    if not cifras_dec:
        return f"{signo}{entero}"
    if resto:
        i = posicion[resto]
        cifras_dec[i:] = ["(", *cifras_dec[i:], ")"]
    return f"{signo}{entero}.{''.join(cifras_dec)}"


for f in (Fraction(1, 7), Fraction(137, 110), Fraction(-19, 6), Fraction(1, 12), Fraction(3, 8), Fraction(22, 7)):
    print(f"{str(f):>7} = {a_decimal(f)}")

for q in range(1, 61):
    for p in range(-2 * q, 2 * q + 1):
        assert a_fraccion(a_decimal(Fraction(p, q))) == Fraction(p, q)


def orden_multiplicativo(a, m):
    k, x = 1, a % m
    while x != 1:
        x = x * a % m
        k += 1
    return k


for q in range(3, 500):
    if math.gcd(q, 10) == 1:
        periodo = a_decimal(Fraction(1, q)).split("(")[1].rstrip(")")
        assert len(periodo) == orden_multiplicativo(10, q)
print("Periodo de 1/97:", len(a_decimal(Fraction(1, 97)).split("(")[1]) - 1, "cifras")

    1/7 = 0.(142857)
137/110 = 1.2(45)
  -19/6 = -3.1(6)
   1/12 = 0.08(3)
    3/8 = 0.375
   22/7 = 3.(142857)
Periodo de 1/97: 96 cifras


## 13. Expresiones con fracciones

Evaluar una suma y resta de fracciones y enteros escrita como texto (`"1/2 + 3/4 - 5/6"`) y dar el resultado simplificado y como número mixto (`43/12` es `3 7/12`). Leer los términos con una expresión regular, sin `eval`, y dar un error claro si la expresión está mal escrita o algún denominador es 0. Comprobarlo con expresiones aleatorias.

In [14]:
TERMINO = re.compile(r"\s*([+-])\s*(\d+)(?:\s*/\s*(\d+))?\s*")


def evalua(expresion):
    expresion = expresion.strip()
    if not expresion.startswith(("+", "-")):
        expresion = "+" + expresion
    total, pos = Fraction(0), 0
    while pos < len(expresion):
        m = TERMINO.match(expresion, pos)
        if not m:
            raise ValueError(f"expresión no válida cerca de {expresion[pos:]!r}")
        signo, numerador, denominador = m.groups()
        if denominador is not None and int(denominador) == 0:
            raise ValueError(f"denominador nulo en {m.group().strip()!r}")
        valor = Fraction(int(numerador), int(denominador or 1))
        total += valor if signo == "+" else -valor
        pos = m.end()
    return total


def mixto(fraccion):
    signo = "-" if fraccion < 0 else ""
    entero, resto = divmod(abs(fraccion.numerator), fraccion.denominator)
    if resto == 0:
        return f"{signo}{entero}"
    parte = f"{resto}/{fraccion.denominator}"
    return f"{signo}{entero} {parte}" if entero else f"{signo}{parte}"


for expresion in ("1/2 + 3/4 - 5/6", "7/3 + 5/4", "-2/3 - 1/6", "10/4 - 1/2", "3 - 22/7", "15/4+2/3-1"):
    r = evalua(expresion)
    print(f"{expresion:>18} = {str(r):>6} = {mixto(r)}")
for mala in ("1/2 + * 3", "1/0 + 2", "1/2 3/4"):
    try:
        evalua(mala)
    except ValueError as e:
        print(f"{mala!r}: {e}")

random.seed(13)
for _ in range(500):
    terminos = [(random.choice("+-"), random.randint(0, 30), random.randint(1, 12)) for _ in range(random.randint(1, 6))]
    texto = " ".join(f"{s} {a}/{b}" for s, a, b in terminos)
    esperado = sum(Fraction(a, b) if s == "+" else -Fraction(a, b) for s, a, b in terminos)
    r = evalua(texto)
    assert r == esperado
    partes = mixto(r).lstrip("-").split()
    assert (-1 if r < 0 else 1) * sum(Fraction(p) for p in partes) == r

   1/2 + 3/4 - 5/6 =   5/12 = 5/12
         7/3 + 5/4 =  43/12 = 3 7/12
        -2/3 - 1/6 =   -5/6 = -5/6
        10/4 - 1/2 =      2 = 2
          3 - 22/7 =   -1/7 = -1/7
        15/4+2/3-1 =  41/12 = 3 5/12
'1/2 + * 3': expresión no válida cerca de '+ * 3'
'1/0 + 2': denominador nulo en '+1/0'
'1/2 3/4': expresión no válida cerca de '3/4'
